[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/mly1101-machine-learning/blob/main/notebooks/12_alumno_crispdm.ipynb)

# MLY1101 · Machine Learning — Actividad 2.1
## Gestión de proyectos con CRISP-DM

**Resultado de aprendizaje (RA2):** aplica modelos estadísticos al conjunto de datos
procesados para interpretarlos, **utilizando metodologías ágiles**, con la finalidad de
obtener conocimientos relevantes que permitan responder a las necesidades del contexto
de negocio, considerando aspectos éticos.

**Indicador de logro (IL 2.1):** implementa metodologías de trabajo (como **CRISP-DM**)
para estructurar el desarrollo del modelo.

---

### La idea central de hoy

El RA1 no empezó eligiendo un algoritmo. Empezó por los datos. Eso ya era CRISP-DM, solo
que nadie lo había nombrado.

Hoy hacemos tres cosas, y ninguna es memorizar siglas:

1. **Ponerle nombre** a lo que ya hicieron (comprensión y preparación de datos).
2. **Cerrar el hueco** que se saltaron: la comprensión del negocio. El problema de las
   detecciones LiDAR se les *entregó*; no lo formularon.
3. **Planificar** el resto del semestre sobre el mismo mapa, para que el informe del EFT
   no se escriba de memoria al final.

```
Comprensión del negocio → Datos → Preparación → Modelado → Evaluación → Despliegue
         └─ hueco de hoy ─┘      └── RA1 ──┘     └── RA2 / RA3 ──┘     └── EFT ──┘
```

> CRISP-DM no es una cascada. Evaluar puede devolverte al negocio ("esta métrica no
> responde la pregunta") o a los datos ("esta partición miente").

---

### Al final de la sesión debes entregar

Una **carta de proyecto CRISP-DM** del hilo Waymo (se valida en código) y la misma carta
rellenada para el **caso oficial** de tu equipo (Telco, House Prices o Spotify), que es
la que viaja a la Parcial 2 y al EFT.

---
## Preparación del entorno

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO = Path("mly1101-machine-learning")
    if not REPO.exists():
        !git clone -q https://github.com/Giocrisrai/mly1101-machine-learning.git {REPO}
    RAIZ = REPO.resolve()
else:
    RAIZ = Path("..").resolve()

sys.path.insert(0, str(RAIZ / "src"))
RUTA_DATOS = RAIZ / "datos" / "crudos" / "detecciones_waymo_like.csv"

print("Colab:", EN_COLAB, "| dataset:", RUTA_DATOS.exists())

In [ ]:
import pandas as pd

import crispdm
import eda

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

df = pd.read_csv(RUTA_DATOS)
print(f"{df.shape[0]:,} detecciones en {df['segment_id'].nunique()} segmentos")
print("fases CRISP-DM:", len(crispdm.FASES))

---
# Bloque 1 · Las seis fases (y por qué no son una cascada)

CRISP-DM (Cross-Industry Standard Process for Data Mining) describe **seis fases**. El
orden importa, pero no es un edificio: es un ciclo. La evaluación puede devolver al
negocio o a los datos.

### ✏️ TODO 1 — El orden del ciclo

Asigna a `orden` las seis claves de `crispdm.FASES` **en el orden del ciclo**. No las
copies de memoria si puedes leer el módulo: parte del trabajo es usar la herramienta,
no recitarla.

In [ ]:
orden = []  # TODO: las seis claves, en orden
print(" → ".join(crispdm.NOMBRES[f] for f in orden))
assert orden == list(crispdm.FASES), "el orden canónico vive en crispdm.FASES"

### ✏️ TODO 2 — ¿Cascada o ciclo?

`crispdm.RETORNOS` declara desde qué fase se puede volver, y a dónde. Imprímelo y
responde: si en la Actividad 2.2 la exactitud sale alta y el F1 de la clase minoritaria
sale bajo, **¿a qué fase vuelves y por qué?**

In [ ]:
for origen, destinos in crispdm.RETORNOS.items():
    print(f"{crispdm.NOMBRES[origen]:<28} → {', '.join(crispdm.NOMBRES[d] for d in destinos)}")

**Si la exactitud es alta y el F1 de la minoría es bajo, vuelvo a:** `____`

**Por qué esa fase y no otra:** `____`

---
# Bloque 2 · ⭐⭐ Retroceso: el RA1 ya era CRISP-DM

Cuatro actividades, 23 horas, y nadie dijo "fase 2". Eso no fue un olvido: fue el
método. Hoy hay que **mapear** lo que ya está hecho para no volver a hacerlo y para
ver el hueco.

Estos hallazgos ya los midieron. Cada uno pertenece a **una** fase principal.

| Hallazgo | Fase (clave de `crispdm.FASES`) |
|---|---|
| El CSV tiene 10 defectos intencionales y 40.680 filas | `____` |
| `CYCLIST` es ~2 % de las filas | `____` |
| La tabla de decisiones de limpieza (qué se imputa, qué se tira) | `____` |
| El censo de Waymo: 793 de 798 segmentos son `sunny` | `____` |
| Parquet conserva tipos; CSV los pierde | `____` |
| Combinaciones de columnas que reidentifican | `____` |

### ✏️ TODO 3 — Completa el mapa en código

In [ ]:
hallazgos = {
    "diez_defectos": "",            # TODO
    "desbalance_cyclist": "",       # TODO
    "tabla_de_decisiones": "",      # TODO
    "censo_sunny": "",              # TODO
    "parquet_vs_csv": "",           # TODO
    "reidentificacion": "",         # TODO
}

fases_validas = set(crispdm.FASES)
assert all(f in fases_validas for f in hallazgos.values()), "usa las claves de crispdm.FASES"
print(pd.Series(hallazgos).rename("fase").to_string())

**La fase que casi no aparece en esa tabla es:** `____`

**Eso significa, en una frase, que el RA1:** `____`

---
# Bloque 3 · ⭐⭐ El negocio que no formularon

En el RA1 el problema llegó escrito: *"trabajas en el equipo de percepción…"*. Eso
ahorra tiempo y **esconde el trabajo**. Un cliente no entrega la pregunta limpia. La
primera fase de CRISP-DM consiste en escribirla de modo que se pueda fallar.

Hay dos maneras de fallar esta fase, y las dos tienen detector en `crispdm`:

1. Preguntar **qué algoritmo** usar. Eso no es una pregunta de negocio.
2. Declarar un criterio de éxito **sin cifra**. Eso es un deseo.

### ✏️ TODO 4 — ¿Esta pregunta empieza por el algoritmo?

Clasifica cada pregunta con `crispdm.empieza_por_el_algoritmo`.

In [ ]:
preguntas = {
    "A": "¿Qué modelo usamos, random forest o red neuronal?",
    "B": "¿En qué condiciones el sensor deja de ser confiable?",
    "C": "Vamos a probar XGBoost porque gana las competencias",
    "D": "¿Se puede anticipar qué detecciones van a ser difíciles?",
}

for clave, texto in preguntas.items():
    marca = "algoritmo" if crispdm.empieza_por_el_algoritmo(texto) else "negocio"
    print(f"{clave}  {marca:10}  {texto}")

**Las que empiezan por el algoritmo son:** `____`
**La pregunta de negocio del equipo de percepción, en una línea, es:** `____`

### ✏️ TODO 5 — Un criterio sin cifra no es un criterio

`crispdm.es_criterio_medible` exige un dígito: un umbral, un porcentaje, una cantidad.
Sin eso, nadie puede decir si el proyecto cumplió.

In [ ]:
criterios = [
    "el modelo tiene que ser bueno",
    "mejorar la percepción del vehículo",
    "F1-macro ≥ 0,70 en detecciones difíciles",
    "recall de LEVEL_2 de al menos 0,60",
    "RMSE menor que 20.000 en el conjunto de prueba",
]
for texto in criterios:
    ok = "medible" if crispdm.es_criterio_medible(texto) else "deseo"
    print(f"{ok:8}  {texto}")

**Tu criterio de éxito para el hilo Waymo (una frase, con cifra):** `____`

No vale la exactitud global. Ya sabes por el RA1 que un modelo que siempre dice
`VEHICLE` acierta ~62 % y es inútil.

---
# Bloque 4 · La carta del proyecto (hilo Waymo)

Una carta de proyecto no es un informe. Es una página que otro equipo podría usar
para continuar sin preguntarte. `crispdm.validar_carta` comprueba cinco campos y dos
trampas: criterio vago y pregunta-algoritmo.

Los campos son: `pregunta_de_negocio`, `criterio_de_exito`, `fuentes`, `riesgos`,
`proxima_fase`.

### ✏️ TODO 6 — Rellena y valida

In [ ]:
carta_waymo = {
    "pregunta_de_negocio": "",   # TODO: una pregunta, no un algoritmo
    "criterio_de_exito": "",     # TODO: con cifra
    "fuentes": "",
    "riesgos": "",               # TODO: con cifras del RA1, no "puede haber sesgo"
    "proxima_fase": "",          # TODO: una clave de crispdm.FASES
}

problemas = crispdm.validar_carta(carta_waymo)
print("problemas:", problemas if problemas else "ninguno — carta usable")
assert problemas == [], problemas

**Riesgo que más puede hundir el criterio de éxito, y con qué cifra:** `____`

---
# Bloque 5 · El mapa del curso (para no repetir el error)

El programa pone el **aprendizaje supervisado y el no supervisado en el RA2**. El RA3
es hiperparámetros, ensamble y validación cruzada. Es el error que más cuesta deshacer
si se planta ahora.

`crispdm.mapa_del_curso()` es la correspondencia oficial de este repositorio.

### ✏️ TODO 7 — ¿Dónde vive el no supervisado?

In [ ]:
mapa = pd.DataFrame(crispdm.mapa_del_curso())
print(mapa.to_string(index=False))
print()
print("RA de la Act. 2.3:", mapa.set_index("actividad").loc["2.3", "ra"])
print("RA de la Act. 3.1:", mapa.set_index("actividad").loc["3.1", "ra"])

**El no supervisado (Act. 2.3) pertenece al:** `____`
**El RA3 no es "la unidad de clustering". El RA3 es:** `____`

### ✏️ TODO 8 — Próximas fases, en una frase cada una

| Actividad | Fase CRISP-DM | Qué pregunta responde (una línea) |
|---|---|---|
| 2.2 Supervisado | `____` | `____` |
| 2.3 No supervisado | `____` | `____` |
| 2.4 Interpretación | `____` | `____` |
| 3.1–3.3 Optimización | `____` | `____` |
| EFT | las seis | `____` |

---
# Bloque 6 · La misma carta, sobre el caso oficial

Las actividades usan detecciones LiDAR. Las **evaluaciones** (Parcial 2, EFT) se rinden
sobre *Telco Customer Churn*, *House Prices* o *Spotify Tracks*. El método tiene que
trasladarse; memorizar el hilo Waymo no basta.

Esta carta **no** la valida el código: el validador no conoce tu caso. La valida la
pauta. Mismas trampas: pregunta-algoritmo y criterio sin cifra.

---

## Carta CRISP-DM — caso oficial

**Equipo:** `____` · **Caso:** Telco / House Prices / Spotify · **Fecha:** `____`

| Campo | Contenido |
|---|---|
| **Pregunta de negocio** | `____` |
| **Criterio de éxito (con cifra)** | `____` |
| **Fuentes** | `____` |
| **Riesgos (con cifra o con columna)** | `____` |
| **Qué ya está hecho del RA1** | `____` |
| **Próxima fase** | `____` |
| **Qué sería "despliegue" en este curso** | `____` |

> Despliegue, aquí, no es un API en producción. El EFT pide informe en Markdown,
> notebook ejecutable, datos para reproducir y estructura de proyecto. Eso **es**
> despliegue a escala de asignatura: otra persona puede correrlo sin preguntarte.

---
### ✅ Antes de cerrar

- [ ] TODO 1–7 en verde (`assert` incluidos).
- [ ] Carta Waymo con `validar_carta` vacío.
- [ ] Carta del caso oficial con pregunta, cifra y riesgos.
- [ ] El no supervisado quedó anotado en el **RA2**, no en el RA3.

### Lo que viene

- **Actividad 2.2:** modelado supervisado. La pregunta de negocio de hoy se vuelve
  un `y` (`detection_difficulty`) y un split por `segment_id`.
- **Actividad 2.3:** modelado no supervisado. Sigue siendo RA2.
- **Actividad 2.4:** evaluación traducida a lenguaje de negocio.